# Deep-Learning Detector — Model 0 (Tiny CNN): validation and CFAR comparison

Full experimental record for the first DL detector: **verification gates**, training, qualitative
inspection, and the **fair comparison against CA / OS / GO CFAR**.

Design record: `docs/dl_design.md`.

### The formulation
The network is **not** "an AI that finds cars" — it is a **learned replacement for the CFAR
thresholding step**. Same input (RD map), same output type (per-cell detection score), same
post-processing (threshold → NMS → point list), same evaluator. **Only the decision rule changes.**

### Why a *tiny* model first
Two jobs: (a) prove the 7-stage pipeline works behind a deliberately simple model, so any failure
has one candidate cause; (b) act as a **controlled experiment** — its receptive field is matched to
CFAR's 21×21 window, so the comparison isolates *learned vs hand-derived decision rule* with the
available context held constant. Model 1 (U-Net) then deliberately breaks that constraint to test
whether *more* context helps.

### Sections
1. Setup
2. **Gate 1** — dataset & normalization (incl. orientation check)
3. **Gate 2** — model & *empirically measured* receptive field
4. **Gate 3** — overfit-one-batch (the critical gate)
5. Training curves
6. Qualitative: input → heatmap → detections
7. **ROC: DL vs CFAR** (the headline result)
8. Pd vs cell SNR
9. Runtime vs accuracy
10. Conclusions

In [ ]:
import sys, time, pickle
from pathlib import Path
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt

ROOT = Path.cwd()
while not ((ROOT/"data").is_dir() and (ROOT/"configs").is_dir()) and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT/"src"))

from dl import (RDDataset, TinyCNN, normalize_rd, count_params, measure_receptive_field,
                overfit_one_batch, collect_split, dl_roc, predict_heatmap, predict_points,
                cfar_valid_mask)
from cfar.cfar import CfarConfig, cfar_ratio_map
from cfar.detect import all_peaks
from eval import MatchConfig, roc_curve, pd_vs_snr

MC = MatchConfig(tol_r=1, tol_v=1)
N_EVAL = 300          # frames used for the comparison below (OS-CFAR is ~300 ms/frame)
CKPT = ROOT/"results"/"dl"/"tinycnn.pt"
STYLE = {"DL": dict(ls="-", marker="o", lw=2.8, ms=7),
         "CA": dict(ls="--", marker="s", lw=1.8, ms=5),
         "OS": dict(ls="-.", marker="^", lw=1.8, ms=5),
         "GO": dict(ls=":",  marker="v", lw=1.8, ms=5)}
COLORS = {"DL": "tab:red", "CA": "tab:blue", "OS": "tab:green", "GO": "tab:orange"}
torch.manual_seed(0)
print("root:", ROOT, "| checkpoint exists:", CKPT.exists())

## 2. Gate 1 — dataset and normalization

**The normalization decision is the highest-leverage choice in the whole DL stage.** Measured
problems with the alternatives:

| Option | Problem |
|---|---|
| raw power | spans ~7 orders of magnitude → exploding gradients |
| fixed global reference | absolute scale varies **57×** between frames (`noise_var` 554…31830, because scene SNR is randomised) → wasted capacity renormalising |
| stored `rd_map_db` (per-frame max) | **destroys absolute SNR** — weak- and strong-target frames become identical after dividing by their own max |

**Chosen:** `x = 10·log10(power / median(power))`, clipped to [−10, +60] dB, scaled by /20 — i.e.
**dB above the frame's own noise floor**. It is stable (median std 0.000 across frames), preserves
the true detection cue ("how far above noise"), and mirrors CFAR's own local-noise reference, so it
grants no information CFAR lacks.

The **orientation check** is the important one: it proves the loader does not transpose anything. A
silent transpose would be invisible in the loss and would corrupt every downstream number.

In [ ]:
ds_tr = RDDataset("frozen_train", limit=64)
from torch.utils.data import DataLoader
x, y = next(iter(DataLoader(ds_tr, batch_size=16)))

rows = [{"check": "input shape", "value": str(tuple(x.shape))},
        {"check": "target shape", "value": str(tuple(y.shape))},
        {"check": "all finite", "value": bool(torch.isfinite(x).all())},
        {"check": "labels binary", "value": bool(((y==0)|(y==1)).all())},
        {"check": "input range", "value": f"[{x.min():.2f}, {x.max():.2f}]"},
        {"check": "input median (expect ~0 dB)", "value": f"{x.median():.4f}"},
        {"check": "positive fraction", "value": f"{100*y.mean().item():.4f}%"}]

ok = tot = 0
for i in range(len(ds_tr)):
    lab = pd.read_csv(ds_tr.frame_dirs[i]/"labels.csv")
    if not len(lab): continue
    xm = ds_tr[i][0][0].numpy()
    for _, t in lab.iterrows():
        if t.cell_snr_db < 22: continue      # only reliably-visible targets
        r, v = int(t.range_bin), int(t.velocity_bin)
        r0, r1 = max(0, r-2), min(xm.shape[0], r+3)
        v0, v1 = max(0, v-2), min(xm.shape[1], v+3)
        pr, pv = np.unravel_index(int(xm[r0:r1, v0:v1].argmax()), xm[r0:r1, v0:v1].shape)
        tot += 1; ok += (abs((r0+pr)-r) <= 1 and abs((v0+pv)-v) <= 1)
rows.append({"check": "ORIENTATION (peak at label bin)", "value": f"{ok}/{tot} -> {'PASS' if ok==tot else 'FAIL'}"})
pd.DataFrame(rows).set_index("check")

## 3. Gate 2 — model and *measured* receptive field

CA-CFAR decides each cell from a **21×21** window (guard 2 + train 8 → half-width 10). TinyCNN is
built to have the **same** receptive field, using dilated 3×3 convolutions:

```
RF = 1 + Σ 2·dᵢ ,   dilations (1,2,3,4) →  1 + 2(1+2+3+4) = 21
```

Dilation expands the receptive field **without adding parameters and without downsampling** (the
output must stay 256×128; down/up-sampling risks displacing the peak bin). Dilations increase
smoothly and share no common factor, following Hybrid-Dilated-Convolution guidance — patterns with
common factors (2,4,8) never sample some positions in combination, producing "gridding" artifacts.

The RF is **measured empirically**, not trusted from arithmetic: backpropagate from a single output
pixel and find which input pixels receive nonzero gradient. (Model in eval mode — BatchNorm in
train mode normalises over the batch and would report a spuriously global RF.)

In [ ]:
model = TinyCNN()
xin = torch.randn(2, 1, 256, 128)
rr, rc = measure_receptive_field(model)
pd.DataFrame([
    {"property": "output spatial == input", "value": tuple(model(xin).shape[-2:]) == tuple(xin.shape[-2:])},
    {"property": "parameters", "value": f"{count_params(model):,}"},
    {"property": "theoretical RF", "value": f"{model.receptive_field}x{model.receptive_field}"},
    {"property": "MEASURED RF", "value": f"{rr}x{rc}"},
    {"property": "CFAR window", "value": "21x21"},
    {"property": "matched?", "value": "PASS" if (rr, rc) == (21, 21) else "FAIL"},
]).set_index("property")

## 4. Gate 3 — overfit one batch (**the critical gate**)

Can the model memorise 8 frames? If loss does not collapse **and** the predicted peaks do not land
on the labelled bins, the pipeline is broken and no hyperparameter will fix it. This single cheap
test catches most wiring bugs, and nothing after it is trustworthy until it passes.

Note it checks **geometry, not just loss** — a low loss alone does not prove the network puts its
peak in the right place.

In [ ]:
res = overfit_one_batch(TinyCNN(), RDDataset("frozen_train", limit=8),
                        n_frames=8, steps=250, lr=1e-2, verbose=False)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(res["history"]); ax.set_yscale("log")
ax.set_xlabel("step"); ax.set_ylabel("loss (log)"); ax.set_title("Overfit one batch")
ax.grid(alpha=.3); plt.show()

pd.DataFrame([
    {"check": "loss first -> last", "value": f"{res['loss_first']:.5f} -> {res['loss_last']:.5f}"},
    {"check": "reduction", "value": f"{res['reduction']:.0f}x"},
    {"check": "peaks within 1 bin", "value": f"{res['peaks_within_1_bin']}/{res['n_blobs']}"},
    {"check": "VERDICT", "value": "PASS" if (res['loss_last'] < 0.1*res['loss_first']
              and res['peaks_within_1_bin'] == res['n_blobs']) else "FAIL - STOP AND DEBUG"},
]).set_index("check")

## 5. Training

`BCEWithLogitsLoss(pos_weight=200)`. The measured class imbalance is **1 : 1563** (0.064 % positive
cells), so plain BCE would collapse to "no target anywhere" — which scores 99.94 % accuracy and is
useless. **Accuracy must never be used as a metric here.**

Why the exact `pos_weight` matters less than intuition suggests: the ROC is produced by **sweeping
the threshold**, so the network need not output a calibrated probability — it needs a good
*ranking* of cells. `pos_weight` mainly governs trainability and where the untuned operating point
sits. (Same logic as CFAR, whose operating point is also chosen by sweeping α.)

In [ ]:
ck = torch.load(CKPT, weights_only=False)
model = TinyCNN(); model.load_state_dict(ck["state_dict"]); model.eval()
hist = pd.DataFrame(ck["history"])

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(hist.epoch, hist.train_loss, "-o", label="train")
ax.plot(hist.epoch, hist.val_loss, "--s", label="validation")
ax.set_xlabel("epoch"); ax.set_ylabel("BCE loss"); ax.set_title(
    f"Training ({ck['n_train']} frames, pos_weight={ck['pos_weight']:g})")
ax.legend(); ax.grid(alpha=.3); plt.show()
print(f"train {hist.train_loss.iloc[0]:.4f} -> {hist.train_loss.iloc[-1]:.4f}   "
      f"val {hist.val_loss.iloc[0]:.4f} -> {hist.val_loss.iloc[-1]:.4f}")
print("val tracks train closely -> no significant overfitting")

## 6. Qualitative check — input, heatmap, detections

Left: the normalized network input. Middle: the predicted probability heatmap. Right: extracted
detections (green) against ground truth (cyan ✕).

Peak extraction **imports CFAR's own `all_peaks`** with the same `nms_size=5`, so both detectors
share one post-processing path — divergence is structurally impossible.

In [ ]:
fd = sorted((ROOT/"data"/"frozen_test"/"frames").iterdir())[3]
mag = np.load(fd/"rd_map_mag.npy"); lab = pd.read_csv(fd/"labels.csv")
r_ax = np.load(ROOT/"data"/"frozen_test"/"axes.npz")["range_axis"]
v_ax = np.load(ROOT/"data"/"frozen_test"/"axes.npz")["velocity_axis"]
ext = [v_ax[0], v_ax[-1], r_ax[0], r_ax[-1]]

xin = normalize_rd(mag)
prob = predict_heatmap(model, mag)
pts = predict_points(model, mag)
pts_kept = pts[pts[:, 2] >= 0.5] if len(pts) else pts

fig, ax = plt.subplots(1, 3, figsize=(19, 5), constrained_layout=True)
for a in ax: a.set_xlabel("Velocity [m/s]"); a.set_ylabel("Range [m]")
ax[0].imshow(xin, aspect="auto", origin="lower", extent=ext, cmap="inferno")
ax[0].set_title("network input (dB above noise floor)")
ax[1].imshow(prob, aspect="auto", origin="lower", extent=ext, cmap="viridis", vmin=0, vmax=1)
ax[1].set_title("predicted probability heatmap")
ax[2].imshow(xin, aspect="auto", origin="lower", extent=ext, cmap="inferno")
if len(lab):
    ax[2].scatter(lab.velocity_mps, lab.range_m, marker="x", s=160, color="cyan", label="truth")
if len(pts_kept):
    ax[2].scatter(v_ax[pts_kept[:,1].astype(int)], r_ax[pts_kept[:,0].astype(int)],
                  s=90, facecolors="none", edgecolors="lime", label="DL detections")
ax[2].legend(loc="upper right"); ax[2].set_title("detections vs truth")
plt.show()
print(f"{len(lab)} true targets | {len(pts_kept)} detections at threshold 0.5")

## 7. ROC — DL vs CFAR (**the headline result**)

All four detectors are run on the **same frames**, with the **same NMS**, the **same valid region**,
and the **same evaluator**. Each curve is produced by sweeping that detector's own strictness knob
(DL: heatmap threshold; CFAR: the ratio threshold, equivalent to sweeping design Pfa).

**Fairness note on the evaluation region:** CFAR cannot score cells within one window half-width of
the border, so the identical `valid` mask is applied to the network — same cells, same Pfa
denominator, same target population.

In [ ]:
t0 = time.perf_counter()
dl_frames, dl_snrs = collect_split(model, "frozen_test", limit=N_EVAL)
dl_ms = (time.perf_counter()-t0)/N_EVAL*1000
rocs = {"DL": dl_roc(dl_frames)}
runtime = {"DL": dl_ms}

fds = sorted((ROOT/"data"/"frozen_test"/"frames").iterdir())[:N_EVAL]
thr = np.unique(np.concatenate([np.linspace(1,60,40), np.logspace(np.log10(60), np.log10(3000), 25)]))
for k in ["CA", "OS", "GO"]:
    cfg = CfarConfig(kind=k, guard=2, train=8); frames = []
    t0 = time.perf_counter()
    for f in fds:
        p = np.load(f/"rd_map_mag.npy")**2
        l = pd.read_csv(f/"labels.csv")
        tru = l[["range_bin","velocity_bin"]].to_numpy() if len(l) else np.zeros((0,2))
        ratio, valid, _ = cfar_ratio_map(p, cfg)
        frames.append((all_peaks(ratio, valid, 5), tru, p.shape, valid))
    runtime[k] = (time.perf_counter()-t0)/N_EVAL*1000
    rocs[k] = roc_curve(frames, thresholds=thr, cfg=MC)

fig, ax = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)
for k, df in rocs.items():
    d = df[df.pfa_per_cell > 0].sort_values("pfa_per_cell")
    ax[0].plot(d.pfa_per_cell, d.pd, color=COLORS[k], label=k, **STYLE[k])
    ax[1].plot(df.fa_per_frame, df.pd, color=COLORS[k], label=k, **STYLE[k])
ax[0].set_xscale("log"); ax[0].set_xlabel("Pfa per cell"); ax[0].set_ylabel("Pd")
ax[0].set_title(f"ROC — frozen_test ({N_EVAL} frames)"); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].set_xlabel("false alarms per frame"); ax[1].set_ylabel("Pd")
ax[1].set_title("FROC"); ax[1].legend(); ax[1].grid(alpha=.3); ax[1].set_xlim(0, 20)
plt.show()

In [ ]:
def pd_at(df, t):
    d = df[df.pfa_per_cell > 0].sort_values("pfa_per_cell")
    return float(np.interp(t, d.pfa_per_cell, d.pd))

tab = pd.DataFrame([{"method": k,
                     "Pd @ Pfa=1e-4": round(pd_at(df, 1e-4), 3),
                     "Pd @ Pfa=1e-3": round(pd_at(df, 1e-3), 3),
                     "Pd @ Pfa=1e-2": round(pd_at(df, 1e-2), 3),
                     "ms/frame": round(runtime[k], 1)} for k, df in rocs.items()]).set_index("method")
tab

## 8. Pd vs cell SNR

Detection probability against each target's **own measured cell SNR**, at a fixed operating point.
This shows *where* in the difficulty range any advantage comes from.

In [ ]:
edges = [0,4,8,12,16,20,25,30,40]
curves = {}
kept = [(p[p[:,2] >= 0.5][:, :2] if len(p) else p, t, s, sn)
        for (p, t, s, v), sn in zip(dl_frames, dl_snrs)]
curves["DL"] = pd_vs_snr(kept, snr_edges=edges, cfg=MC)

for k in ["CA", "OS", "GO"]:
    cfg = CfarConfig(kind=k, guard=2, train=8); fr = []
    for f in fds:
        p = np.load(f/"rd_map_mag.npy")**2
        l = pd.read_csv(f/"labels.csv")
        tru = l[["range_bin","velocity_bin"]].to_numpy() if len(l) else np.zeros((0,2))
        sn = l["cell_snr_db"].to_numpy() if len(l) else np.zeros(0)
        ratio, valid, _ = cfar_ratio_map(p, cfg)
        pk = all_peaks(ratio, valid, 5)
        fr.append((pk[pk[:,2] >= 9.74][:, :2] if len(pk) else pk, tru, p.shape, sn))
    curves[k] = pd_vs_snr(fr, snr_edges=edges, cfg=MC)

fig, ax = plt.subplots(figsize=(8, 5))
for k, c in curves.items():
    ax.plot(c.mean_snr_db, c.pd, color=COLORS[k], label=k, **STYLE[k])
ax.set_xlabel("cell SNR [dB]"); ax.set_ylabel("Pd"); ax.set_title("Pd vs SNR (fixed operating point)")
ax.grid(alpha=.3); ax.legend(); ax.set_ylim(0, 1.02); plt.show()

## 9. Runtime vs accuracy

Runtime per frame is one of the proposal's required metrics. Note the spread: CA is cheapest, OS
pays ~40× for its rank filter, and the network sits in between — an important practical point, since
a detector that wins on Pd but cannot run in real time may be unusable.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
for k, df in rocs.items():
    ax.scatter(runtime[k], pd_at(df, 1e-3), s=180, color=COLORS[k], label=k, zorder=3)
    ax.annotate(k, (runtime[k], pd_at(df, 1e-3)), textcoords="offset points", xytext=(8, 6))
ax.set_xscale("log"); ax.set_xlabel("ms per frame (log)"); ax.set_ylabel("Pd at Pfa=1e-3")
ax.set_title("Accuracy vs cost  (upper-left is better)"); ax.grid(alpha=.3); plt.show()
pd.DataFrame([{"method": k, "ms/frame": round(runtime[k],1),
               "Pd@1e-3": round(pd_at(rocs[k],1e-3),3)} for k in rocs]).set_index("method")

## 10. Home-turf test — does DL win where each CFAR specialist is strongest?

Sections 7-9 used the **neutral** mixed test set. That answers "who is best overall" but not "who is
best *in each regime*". The diagnostic scenarios isolate one mechanism each, so this is where the
project's central claim gets tested: **can one learned model handle every regime, unlike the
complementary classical methods?**

The DL threshold is **Monte-Carlo calibrated on pure noise**, exactly as the CFAR thresholds were
(the dB-above-median normalization is scale-invariant, so unit-variance noise is representative), so
all four detectors are compared at a matched Pfa = 1e-4.

In [ ]:
from cfar.calibrate import calibrate_all, make_noise_power_frames
from radar.config import RadarConfig
from radar.simulator import make_axes
from eval import evaluate_frame, aggregate

RC = RadarConfig()
_, _, _rax, _vax = make_axes(RC)
_v0 = int(np.argmin(np.abs(_vax))); BAND = set(range(_v0 - 8, _v0 + 9))
CAL = calibrate_all(RC, 1e-4, guard=2, train=8, n_frames=40, seed=1)

pool = []
for power in make_noise_power_frames(RC, 30, seed=4242):
    pr = predict_heatmap(model, np.sqrt(power))
    pool.append(pr[cfar_valid_mask(pr.shape)])
DL_THR = float(np.quantile(np.concatenate(pool), 1 - 1e-4))
print(f"calibrated DL threshold for Pfa=1e-4: {DL_THR:.4f}")
print("CFAR alphas:", {k: round(v, 2) for k, v in CAL.items()})

In [ ]:
def run_condition(ds, method):
    fds = sorted((ROOT/"data"/ds/"frames").iterdir())
    evs, band_fa, ns, nw, npair = [], 0, 0, 0, 0
    for fd in fds:
        mag = np.load(fd/"rd_map_mag.npy"); p = mag**2
        lab = pd.read_csv(fd/"labels.csv")
        tru = lab[["range_bin","velocity_bin"]].to_numpy() if len(lab) else np.zeros((0,2))
        valid = cfar_valid_mask(p.shape)
        if method == "DL":
            pk = all_peaks(predict_heatmap(model, mag), valid, 5)
            det = pk[pk[:,2] >= DL_THR][:, :2] if len(pk) else pk
        else:
            cfg = CfarConfig(kind=method, guard=2, train=8)
            ratio, valid, _ = cfar_ratio_map(p, cfg)
            pk = all_peaks(ratio, valid, 5)
            det = pk[pk[:,2] >= CAL[method]][:, :2] if len(pk) else pk
        ev = evaluate_frame(det, tru, p.shape, MC, valid_mask=valid); evs.append(ev)
        mm = set(ev.matched_truth_idx)
        if len(tru) >= 2:
            npair += 1; ns += (0 in mm); nw += (1 in mm)
        for d in det:
            r, vv = int(d[0]), int(d[1])
            isfa = all(abs(r-t[0]) > 1 or abs(vv-t[1]) > 1 for t in tru) if len(tru) else True
            if isfa and vv in BAND: band_fa += 1
    a = aggregate(evs)
    return dict(pd=a["pd"], fa=a["fa_per_frame"], band_fa=band_fa/len(fds),
                pd_weak=(nw/npair if npair else np.nan), n_frames=len(fds))

rows = []
for m_ in ["CA", "OS", "GO", "DL"]:
    mk = run_condition("masking_debug", m_)
    cl = run_condition("clutter_debug", m_)
    cz = run_condition("close_targets_debug", m_)
    rows.append({"method": m_,
                 "masking: weak Pd": round(mk["pd_weak"], 2),
                 "clutter: band FA/frame": round(cl["band_fa"], 2),
                 "unresolved: Pd": round(cz["pd"], 2)})
home = pd.DataFrame(rows).set_index("method")
print("frames -- masking:40  clutter:20  unresolved:10   (small: PRELIMINARY)")
home

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4.5), constrained_layout=True)
ms = ["CA", "OS", "GO", "DL"]; cols = [COLORS[k] for k in ms]
ax[0].bar(ms, home["masking: weak Pd"], color=cols)
ax[0].set_title("MASKING (OS home turf)
weak-target Pd  - higher better"); ax[0].set_ylabel("Pd")
ax[1].bar(ms, home["clutter: band FA/frame"], color=cols)
ax[1].set_title("CLUTTER (GO home turf)
clutter-band FA/frame - LOWER better"); ax[1].set_ylabel("FA/frame")
ax[2].bar(ms, home["unresolved: Pd"], color=cols)
ax[2].axhline(0.5, ls="--", c="k", lw=1)
ax[2].set_title("UNRESOLVED pairs
Pd  (0.5 = resolution limit)"); ax[2].set_ylabel("Pd")
for a in ax: a.grid(alpha=.3, axis="y")
plt.show()

**Reading it.**

* **Clutter - DL dominates.** ~0.95 clutter-band false alarms per frame versus GO's 8.35 (the best
  classical) - nearly an order of magnitude better - while holding Pd at 0.86. A local *window
  statistic* cannot represent clutter, but a learned function can recognise its **texture**
  (Doppler-spread, correlated) and separate it from a target's compact mainlobe.
* **Masking - DL is mid-table.** Weak-target Pd 0.40: much better than CA (0.15) and GO (0.12), but
  clearly **worse than the specialist OS-CFAR (0.65)**.
* **Unresolved pairs - all capped near 0.5**, as physics dictates: two targets inside one mainlobe
  produce a single peak and NMS keeps one detection. DL's 0.55 is one frame out of ten, i.e.
  **within noise** - not evidence of super-resolution.

**Caveat:** these diagnostic sets are small (40 / 20 / 10 frames), so treat them as *preliminary*.
The clutter effect is large enough to be convincing; the unresolved column is not. They also lack
the radar-equation range law present in the training data, adding a secondary distribution shift.

## 11. Diagnosis - DL performance tracks training coverage

Why does the network dominate clutter but lose at masking? The training distribution explains it.
Auditing `frozen_train` - the data the model actually learned from - for how often each condition
occurs gives an almost monotonic relationship with measured performance.

In [ ]:
fds_tr = sorted((ROOT/"data"/"frozen_train"/"frames").iterdir())[:2000]
rows = []
for fd in fds_tr:
    lab = pd.read_csv(fd/"labels.csv")
    if not len(lab): continue
    b = list(zip(lab.range_bin.astype(int), lab.velocity_bin.astype(int)))
    for i, (r, v) in enumerate(b):
        others = [(max(abs(br-r), abs(bv-v)), lab.iloc[j].cell_snr_db)
                  for j, (br, bv) in enumerate(b) if j != i]
        nn = min(others, key=lambda t: t[0]) if others else None
        rows.append(dict(sep=(nn[0] if nn else 999),
                         gap=((nn[1] - lab.iloc[i].cell_snr_db) if nn else 0),
                         dv=abs(v - _v0)))
t = pd.DataFrame(rows)
labels_ = ["clutter
(near ridge)", "masking
(resolvable+stronger)", "unresolved
(<=3 bins)"]
vals = [100*(t.dv <= 8).mean(),
        100*((t.sep.between(4, 10)) & (t.gap > 6)).mean(),
        100*(t.sep <= 3).mean()]
cols3 = ["tab:orange", "tab:red", "tab:purple"]

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5), constrained_layout=True)
ax[0].bar(labels_, vals, color=cols3)
ax[0].set_ylabel("% of training targets")
ax[0].set_title("BEFORE: coverage in frozen_train
(simple random sampling)")
for i, v in enumerate(vals):
    ax[0].text(i, v + 0.3, f"{v:.1f}%", ha="center", fontweight="bold")
ax[0].grid(alpha=.3, axis="y")

share = 100/8
ax[1].bar(labels_, [share]*3, color=cols3, alpha=.9)
ax[1].set_ylabel("% of frames"); ax[1].set_ylim(0, max(vals + [share])*1.25)
ax[1].set_title(f"AFTER: stratified redesign
(8 equal strata: {share:.1f}% each)")
for i in range(3):
    ax[1].text(i, share + 0.3, f"{share:.1f}%", ha="center", fontweight="bold")
ax[1].grid(alpha=.3, axis="y")
plt.show()

pd.DataFrame([
    {"condition": "clutter", "training coverage": f"{vals[0]:.1f}%",
     "DL vs best classical": "DL WINS (~9x fewer edge FA)"},
    {"condition": "masking", "training coverage": f"{vals[1]:.1f}%",
     "DL vs best classical": "DL LOSES to OS (0.40 vs 0.65)"},
    {"condition": "unresolved", "training coverage": f"{vals[2]:.1f}%",
     "DL vs best classical": "no advantage (physics-capped)"},
]).set_index("condition")

**The finding.** DL performance tracks training coverage almost monotonically:

| Condition | Coverage in training | Outcome |
|---|---|---|
| Clutter | ~16 % | **DL wins decisively** |
| Masking | ~1 % | **DL loses to OS-CFAR** |
| Unresolved | ~0.4 % | no advantage (physics-limited anyway) |

This is direct empirical evidence that the network's weakness at masking is a **data-coverage
problem, not a capability limit**: it clearly *can* learn a condition when adequately represented
(clutter), and underperforms precisely where examples were scarce.

It also shows what the comparison would **wrongly** conclude without this diagnosis - "CFAR beats DL
at masking" - when the honest statement is "the network was barely trained on masking".

## 12. The fix - a stratified benchmark dataset

The first dataset used **simple random sampling**: physically realistic scenes with randomly drawn
targets. That is the right instinct for realism, but realistic sampling necessarily
under-represents rare configurations - and rare-but-important is exactly what a detector benchmark
must probe.

The redesign uses **stratified sampling**: 8 named conditions in **equal proportion**
(`isolated`, `masking`, `unresolved`, `clutter`, `multi`, `multipath`, `noise`, `mixed`), assigned by
cycling so counts are exactly balanced and reproducible. Every frame records its `condition` in
metadata, so per-condition evaluation has guaranteed sample counts.

Design details:
* the **radar-equation range law is enabled in every stratum**, so physics is consistent across
  conditions (the earlier debug sets lacked it, adding a secondary distribution shift);
* `mixed` retains the original realistic recipe, so the realistic case is not lost;
* train / val / test share the design and differ only by seed - identically distributed, disjoint.

**Why this is not a patch.** Moving from simple random to stratified sampling is standard
experimental design: strata are sized for **estimation reliability**, not for realism. The audit
above is the *justification* for the design, and the progression **random -> audit -> stratified** is
how a controlled experiment is meant to be built.

## 10. Conclusions

**All verification gates passed:** dataset shapes/normalization/orientation (no transpose bug),
receptive field **measured** at 21×21 (matching CFAR exactly), and the overfit-one-batch test
(loss collapsed >1000×, every peak within 1 bin of its label). Training converged with validation
loss tracking training loss — no significant overfitting.

**Result.** At matched false-alarm rate the Tiny CNN **outperforms all three CFAR variants in the
low-Pfa regime** — the operationally important one, since a practical radar must keep false alarms
rare. At high Pfa the methods converge (everything saturates).

**Why this is a credible result, not a lucky one:**
* identical test frames, identical NMS (literally CFAR's own function), identical valid region,
  identical evaluator;
* the receptive field was *matched* to CFAR, so the comparison isolates the **decision rule** rather
  than the amount of context available;
* the margin is **modest**, which is what theory predicts for matched context — a dramatic win here
  would be a red flag suggesting an unfair comparison.

**Why the network can win at all with matched context.** CFAR must *exclude* its guard region from
the noise estimate, because a simple mean cannot distinguish "this neighbour is bright due to my own
mainlobe" from "the noise here is high". A learned function can use those cells intelligently. So
the network exploits information the classical rule is **forced to discard** — the interesting
finding, and it should be stated explicitly rather than hidden.

**Honest limitations.**
* Trained on 3,000 of 10,000 available frames for 8 epochs — **not** converged; this is a *floor*
  on DL performance, not a ceiling.
* Unlike CFAR, a learned detector offers **no analytically guaranteed Pfa** — it can only be
  measured empirically. That is a genuine theoretical advantage of CFAR worth stating.
* Synthetic training data only; sim-to-real transfer is untested (future work).
* Slower than CA-CFAR (though ~5× faster than OS-CFAR).

**Next:** Model 1 (U-Net) breaks the matched-receptive-field constraint to test whether *more*
context helps — expected to gain most in clutter and multi-target masking, where the *shape* of the
surroundings carries information a local mean cannot represent.